# Enhanced ML Analysis of Tennessee School Letter Grades

**Key improvement over original analysis:** We remove all formula-input features (achievement scores, growth scores, success rates, CCR rates) that directly compute the letter grade. Instead, we predict letter grades from **structural and contextual factors** — demographics, teacher quality, discipline, absenteeism, expenditures, and staffing.

**Research question:** What structural conditions predict a school's letter grade, independent of test performance?

**Data:** 2022-23 and 2023-24 school years, all same-year aligned from TN DOE.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

## 1. Data Loading & Cleaning

Load all datasets for both years, clean suppressed values, and prepare for merging.

In [ ]:
def clean_suppressed(val):
    """Convert TN DOE suppressed values to NaN."""
    if isinstance(val, str):
        val_lower = val.strip().lower()
        if any(x in val_lower for x in ['less than', 'fewer than', '**', '*', 'suppressed', 'n/a', '-']):
            return np.nan
        val_clean = val.replace('%', '').strip()
        try:
            return float(val_clean)
        except ValueError:
            return np.nan
    return val


def load_letter_grades(year_dir, year_label):
    """Load letter grades, keep only target + identifiers (no formula inputs)."""
    df = pd.read_excel(f'{year_dir}/letter_grades.xlsx')
    df = df[df['lg_ineligible'] == 0].copy()
    # Filter out non-standard grade values
    df = df[df['lg_grade'].isin(['A', 'B', 'C', 'D', 'F'])].copy()
    keep_cols = ['system', 'system_name', 'school', 'school_name',
                 'school_pool', 'grade_band_3-5', 'grade_band_6-8', 'grade_band_9-12',
                 'lg_grade']
    df = df[keep_cols].copy()
    df['year'] = year_label
    return df


def load_demographics(year_dir, year_label):
    """Load school profile demographics."""
    df = pd.read_excel(f'{year_dir}/school_profile.xlsx', sheet_name='By Student Group')
    rename_map = {
        'district_id': 'system', 'district_no': 'system',
        'school_id': 'school', 'school_no': 'school'
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})
    # Focus on demographics with reasonable coverage (>40% across both years)
    demo_cols = ['african_american_pct', 'hispanic_pct', 'white_pct',
                 'economically_disadvantaged_pct', 'limited_english_proficient_pct',
                 'students_with_disabilities_pct',
                 'black_hispanic_native_american_pct']
    available_demo = [c for c in demo_cols if c in df.columns]
    df_out = df[['system', 'school', 'total'] + available_demo].copy()
    for col in available_demo + ['total']:
        df_out[col] = df_out[col].apply(clean_suppressed)
    df_out.rename(columns={'total': 'enrollment'}, inplace=True)
    return df_out


def load_chronic_absenteeism(year_dir):
    """Load chronic absenteeism — All Students, All Grades only."""
    df = pd.read_excel(f'{year_dir}/chronic_absenteeism.xlsx')
    df = df[(df['student_group'] == 'All Students') & (df['grade_band'] == 'All Grades')].copy()
    df['pct_chronically_absent'] = df['pct_chronically_absent'].apply(clean_suppressed)
    return df[['system', 'school', 'pct_chronically_absent']].copy()


def load_discipline(year_dir, year_label):
    """Load discipline data — school-level overall rates."""
    sheet = 'School' if year_label == '2022-23' else 'school'
    df = pd.read_excel(f'{year_dir}/discipline.xlsx', sheet_name=sheet)
    df = df.rename(columns={'District': 'system', 'School': 'school'})
    # Focus on pct_disciplined (best coverage) and pct_suspended
    disc_cols = ['Percent Suspended', 'Percent Disciplined']
    for col in disc_cols:
        if col in df.columns:
            df[col] = df[col].apply(clean_suppressed)
    rename_disc = {
        'Percent Suspended': 'pct_suspended',
        'Percent Disciplined': 'pct_disciplined'
    }
    df = df.rename(columns=rename_disc)
    keep = ['system', 'school'] + [v for v in rename_disc.values() if v in df.columns]
    return df[keep].copy()


def load_educator_experience(year_dir, year_label):
    """Load educator experience — pivot from long to wide."""
    sheet = f'School Data {year_label}'
    df = pd.read_excel(f'{year_dir}/educator_experience.xlsx', sheet_name=sheet)
    df = df.rename(columns={'DistrictNumber': 'system', 'SchoolNumber': 'school'})
    df['Percentage'] = df['Percentage'].apply(clean_suppressed)
    cats_of_interest = {
        'Experienced Teachers': 'pct_experienced_teachers',
        'Inexperienced Teachers': 'pct_inexperienced_teachers',
        'Teachers with Emergency / Provisional Credentials': 'pct_emergency_credentials',
        'Teachers Teaching Out of Field': 'pct_out_of_field'
    }
    df = df[df['Category'].isin(cats_of_interest.keys())].copy()
    df['Category'] = df['Category'].map(cats_of_interest)
    pivot = df.pivot_table(index=['system', 'school'], columns='Category',
                           values='Percentage', aggfunc='first').reset_index()
    pivot.columns.name = None
    return pivot


def load_teacher_retention(year_dir):
    """Load teacher retention rates."""
    df = pd.read_excel(f'{year_dir}/teacher_retention.xlsx', sheet_name='Teacher Retention')
    df = df.rename(columns={
        'System Number': 'system', 'School Number': 'school',
        'Percent Retained': 'pct_teacher_retained'
    })
    df = df[(df['system'] > 0) & (df['school'] > 0)].copy()
    df['pct_teacher_retained'] = df['pct_teacher_retained'].apply(clean_suppressed)
    return df[['system', 'school', 'pct_teacher_retained']].copy()


def load_staff(year_dir):
    """Load staff data — student-teacher ratio for Teachers."""
    df = pd.read_excel(f'{year_dir}/staff.xlsx', sheet_name='Staffing')
    df = df.rename(columns={
        'System Number': 'system', 'School Number': 'school',
        'Student-Educator Ratio': 'student_teacher_ratio',
        'Educator Count': 'teacher_count'
    })
    df = df[(df['Staff Type'] == 'Teacher') & (df['system'] > 0) & (df['school'] > 0)].copy()
    for col in ['student_teacher_ratio', 'teacher_count']:
        df[col] = df[col].apply(clean_suppressed)
    return df[['system', 'school', 'student_teacher_ratio', 'teacher_count']].copy()


def load_finance(year_dir, year_label):
    """Load per-pupil expenditure (school-level) and funding mix (district-level)."""
    df = pd.read_excel(f'{year_dir}/finance.xlsx')
    # Standardize column names
    rename_map = {}
    for c in df.columns:
        if c.startswith('Key'):
            rename_map[c] = 'Key'
    df = df.rename(columns=rename_map)
    df = df.rename(columns={'Dist': 'system_raw', 'District ID': 'district_id', 'School ID': 'school'})
    if 'district_id' in df.columns:
        df['system'] = df['district_id'].fillna(df['system_raw'])
    else:
        df['system'] = df['system_raw']
    df['system'] = pd.to_numeric(df['system'], errors='coerce').astype('Int64')
    df['school'] = pd.to_numeric(df['school'], errors='coerce').astype('Int64')
    
    # District-level funding percentages (only on 9999 rows)
    district_funding = df[df['school'] == 9999][['system', 'Federal Percentage', 'State Percentage', 'Local Percentage']].copy()
    district_funding = district_funding.rename(columns={
        'Federal Percentage': 'pct_federal_funding',
        'State Percentage': 'pct_state_funding',
        'Local Percentage': 'pct_local_funding'
    })
    # Convert to percentages (they're stored as decimals)
    for col in ['pct_federal_funding', 'pct_state_funding', 'pct_local_funding']:
        district_funding[col] = pd.to_numeric(district_funding[col], errors='coerce') * 100
    
    # School-level PPE
    school_data = df[df['school'] != 9999].copy()
    school_data = school_data[school_data['school'].notna()].copy()
    ppe_cols = {
        'Total School Level Per Pupil Expenditures': 'school_ppe',
        'Total District Level Per Pupil Expenditures': 'district_ppe',
        'Total School Per Pupil Expenditures': 'total_ppe',
    }
    available = {k: v for k, v in ppe_cols.items() if k in school_data.columns}
    school_data = school_data.rename(columns=available)
    for col in available.values():
        school_data[col] = pd.to_numeric(school_data[col], errors='coerce')
    
    # Merge district funding onto school rows
    result = school_data[['system', 'school'] + list(available.values())].merge(
        district_funding, on='system', how='left'
    )
    return result


def load_graduation(year_dir):
    """Load graduation rates — All Students only."""
    df = pd.read_excel(f'{year_dir}/graduation.xlsx')
    df = df[df['student_group'] == 'All Students'].copy()
    cohort_col = 'grad_cohort' if 'grad_cohort' in df.columns else 'grad_cohort_state'
    for col in ['grad_rate_state', cohort_col]:
        if col in df.columns:
            df[col] = df[col].apply(clean_suppressed)
    df = df.rename(columns={cohort_col: 'grad_cohort'})
    return df[['system', 'school', 'grad_rate_state', 'grad_cohort']].copy()


def load_dropout(year_dir):
    """Load dropout rates — All Students only."""
    df = pd.read_excel(f'{year_dir}/dropout.xlsx')
    df = df[df['student_group'] == 'All Students'].copy()
    df['dropout_rate'] = df['dropout_rate'].apply(clean_suppressed)
    return df[['system', 'school', 'dropout_rate']].copy()


print('All loading functions defined.')

In [ ]:
def build_year(year_label):
    """Build the full merged dataset for one year."""
    year_dir = f'data/{year_label}'
    
    # Load base (letter grades with no formula inputs)
    df = load_letter_grades(year_dir, year_label)
    n_start = len(df)
    print(f'\n{year_label}: {n_start} eligible schools')
    
    # Merge each dataset
    datasets = {
        'demographics': load_demographics(year_dir, year_label),
        'absenteeism': load_chronic_absenteeism(year_dir),
        'discipline': load_discipline(year_dir, year_label),
        'educator_exp': load_educator_experience(year_dir, year_label),
        'retention': load_teacher_retention(year_dir),
        'staff': load_staff(year_dir),
        'finance': load_finance(year_dir, year_label),
        'graduation': load_graduation(year_dir),
        'dropout': load_dropout(year_dir),
    }
    
    for name, right_df in datasets.items():
        before = len(df)
        df = df.merge(right_df, on=['system', 'school'], how='left')
        matched = df[right_df.columns[-1]].notna().sum()
        print(f'  + {name}: {matched}/{before} matched ({matched/before*100:.1f}%)')
    
    return df


# Build both years
df_22 = build_year('2022-23')
df_23 = build_year('2023-24')

# Combine
df = pd.concat([df_22, df_23], ignore_index=True)
print(f'\nCombined: {len(df)} schools across 2 years')
print(f'Columns: {len(df.columns)}')
print(f'\nGrade distribution:')
print(df['lg_grade'].value_counts().sort_index())

In [ ]:
# Inspect the merged dataset
print('Feature columns (excluding identifiers and target):')
id_cols = ['system', 'system_name', 'school', 'school_name', 'year']
target = 'lg_grade'
feature_cols = [c for c in df.columns if c not in id_cols + [target]]
for i, col in enumerate(feature_cols):
    non_null = df[col].notna().sum()
    pct = non_null / len(df) * 100
    print(f'  {i+1:2d}. {col:40s} {non_null:5d}/{len(df)} ({pct:5.1f}%) non-null')

print(f'\nTotal features: {len(feature_cols)}')

In [ ]:
# Handle grade bands — convert Y/N to 1/0
for col in ['grade_band_3-5', 'grade_band_6-8', 'grade_band_9-12']:
    df[col] = (df[col] == 'Y').astype(int)

# Encode school_pool as dummies
pool_dummies = pd.get_dummies(df['school_pool'], prefix='pool', dtype=int)
df = pd.concat([df, pool_dummies], axis=1)

# Encode target as ordinal: F=0, D=1, C=2, B=3, A=4
grade_map = {'F': 0, 'D': 1, 'C': 2, 'B': 3, 'A': 4}
df['lg_grade_ord'] = df['lg_grade'].map(grade_map)

# Add year as numeric
df['year_num'] = df['year'].map({'2022-23': 0, '2023-24': 1})

print('Encoding complete.')
print(f'School pool categories: {df["school_pool"].unique()}')
print(f'Grade ordinal mapping: {grade_map}')

## 2. Exploratory Data Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grade distribution by year
grade_order = ['A', 'B', 'C', 'D', 'F']
ct = pd.crosstab(df['lg_grade'], df['year'])
ct = ct.reindex(grade_order)
ct.plot(kind='bar', ax=axes[0], color=['#2E5599', '#C9A84C'])
axes[0].set_title('Letter Grade Distribution by Year')
axes[0].set_xlabel('Letter Grade')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# School pool distribution
pool_ct = pd.crosstab(df['school_pool'], df['lg_grade'])
pool_ct = pool_ct[grade_order]
pool_ct.plot(kind='bar', stacked=True, ax=axes[1],
             color=['#1a9641', '#a6d96a', '#ffffbf', '#fdae61', '#d7191c'])
axes[1].set_title('Grade Distribution by School Pool')
axes[1].set_xlabel('School Pool')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('figures/grade_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Key contextual features by letter grade
context_features = [
    'economically_disadvantaged_pct', 'pct_chronically_absent',
    'pct_disciplined', 'pct_inexperienced_teachers',
    'pct_teacher_retained', 'student_teacher_ratio',
    'total_ppe'
]
available_context = [c for c in context_features if c in df.columns]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(available_context):
    if i >= len(axes):
        break
    data = df[df[col].notna()]
    sns.boxplot(data=data, x='lg_grade', y=col, order=grade_order,
                palette=['#1a9641', '#a6d96a', '#ffffbf', '#fdae61', '#d7191c'],
                ax=axes[i])
    axes[i].set_title(col.replace('_', ' ').title())
    axes[i].set_xlabel('Letter Grade')

# Hide unused axes
for j in range(len(available_context), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Contextual Features by Letter Grade', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('figures/features_by_grade.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap of numeric features
numeric_features = df[available_context + ['lg_grade_ord']].dropna()
corr = numeric_features.corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=ax, vmin=-1, vmax=1)
ax.set_title('Feature Correlations with Letter Grade (Ordinal)')
plt.tight_layout()
plt.savefig('figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary stats by grade
summary = df.groupby('lg_grade')[available_context].agg(['mean', 'median']).round(2)
summary = summary.reindex(grade_order)
print('Mean values by letter grade:')
means = df.groupby('lg_grade')[available_context].mean().round(2).reindex(grade_order)
print(means.to_string())

## 3. Feature Preparation for Modeling

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold

# Define feature set — everything except identifiers, target, and formula inputs
exclude = ['system', 'system_name', 'school', 'school_name', 'year',
           'lg_grade', 'lg_grade_ord', 'school_pool']
feature_names = [c for c in df.columns if c not in exclude]

print(f'Features for modeling ({len(feature_names)}):')
for f in feature_names:
    print(f'  {f}')

X = df[feature_names].copy()
y = df['lg_grade_ord'].copy()

# Drop rows where target is missing
valid = y.notna()
X = X[valid]
y = y[valid].astype(int)

print(f'\nSamples: {len(X)}')
print(f'Missing values per feature:')
print(X.isnull().sum().sort_values(ascending=False).head(15))

In [ ]:
# Impute missing values with median (robust to suppressed demographic data)
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=feature_names, index=X.index)

# Scale for models that need it (logistic regression, SVM, neural nets)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_imputed), columns=feature_names, index=X.index)

# Train/test split (stratified by grade)
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42, stratify=y
)
X_train_sc, X_test_sc, _, _ = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {len(X_train)}, Test: {len(X_test)}')
print(f'\nTrain grade distribution:')
print(y_train.value_counts().sort_index())
print(f'\nTest grade distribution:')
print(y_test.value_counts().sort_index())

## 4. Model Training & Evaluation

We train multiple models and compare. Since letter grades are ordinal (F < D < C < B < A), we include ordinal-aware approaches alongside standard classifiers.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, mean_absolute_error)
from xgboost import XGBClassifier

# Results storage
results = {}

def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    """Train, predict, and store results."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    mae = mean_absolute_error(y_te, y_pred)  # Ordinal: how many grades off?
    
    # Cross-validation
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_tr, y_tr, cv=cv, scoring='accuracy')
    
    results[name] = {
        'accuracy': acc,
        'mae': mae,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'model': model,
        'y_pred': y_pred
    }
    
    print(f'\n{"="*60}')
    print(f'{name}')
    print(f'{"="*60}')
    print(f'Test Accuracy: {acc:.4f}')
    print(f'Mean Absolute Error: {mae:.4f} (avg grades off)')
    print(f'CV Accuracy: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')
    print(f'\nClassification Report:')
    grade_names = ['F', 'D', 'C', 'B', 'A']
    print(classification_report(y_te, y_pred, target_names=grade_names))
    return model

In [ ]:
# 1. Random Forest
rf = RandomForestClassifier(
    n_estimators=500, max_depth=None, min_samples_leaf=5,
    random_state=42, n_jobs=-1
)
evaluate_model('Random Forest', rf, X_train, X_test, y_train, y_test)

In [ ]:
# 2. XGBoost
xgb = XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    objective='multi:softmax', num_class=5,
    random_state=42, n_jobs=-1, verbosity=0
)
evaluate_model('XGBoost', xgb, X_train, X_test, y_train, y_test)

In [ ]:
# 3. Gradient Boosting
gb = GradientBoostingClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.1,
    subsample=0.8, random_state=42
)
evaluate_model('Gradient Boosting', gb, X_train, X_test, y_train, y_test)

In [ ]:
# 4. Logistic Regression (ordinal-adjacent: multinomial)
lr = LogisticRegression(
    multi_class='multinomial', max_iter=2000,
    C=1.0, random_state=42
)
evaluate_model('Logistic Regression', lr, X_train_sc, X_test_sc, y_train, y_test)

In [ ]:
# 5. Ordinal Logistic Regression (proportional odds model)
# This respects the ordering F < D < C < B < A
from sklearn.base import BaseEstimator, ClassifierMixin

class OrdinalClassifier(BaseEstimator, ClassifierMixin):
    """Ordinal classification via cumulative logistic regression."""
    def __init__(self, max_iter=2000, C=1.0):
        self.max_iter = max_iter
        self.C = C
        self.classifiers_ = []
        self.classes_ = None
    
    def fit(self, X, y):
        self.classes_ = np.sort(np.unique(y))
        self.classifiers_ = []
        for i in range(len(self.classes_) - 1):
            binary_y = (y > self.classes_[i]).astype(int)
            clf = LogisticRegression(max_iter=self.max_iter, C=self.C, random_state=42)
            clf.fit(X, binary_y)
            self.classifiers_.append(clf)
        return self
    
    def predict_proba(self, X):
        probs = np.zeros((len(X), len(self.classes_)))
        cum_probs = np.column_stack([
            clf.predict_proba(X)[:, 1] for clf in self.classifiers_
        ])
        probs[:, 0] = 1 - cum_probs[:, 0]
        for i in range(1, len(self.classes_) - 1):
            probs[:, i] = cum_probs[:, i-1] - cum_probs[:, i]
        probs[:, -1] = cum_probs[:, -1]
        probs = np.clip(probs, 0, 1)
        probs = probs / probs.sum(axis=1, keepdims=True)
        return probs
    
    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

ordinal = OrdinalClassifier(max_iter=2000, C=1.0)
evaluate_model('Ordinal Logistic', ordinal, X_train_sc, X_test_sc, y_train, y_test)

In [ ]:
# Model comparison summary
print(f'{"Model":<25s} {"Accuracy":>10s} {"MAE":>8s} {"CV Mean":>10s} {"CV Std":>8s}')
print('-' * 65)
for name, r in sorted(results.items(), key=lambda x: -x[1]['accuracy']):
    print(f'{name:<25s} {r["accuracy"]:>10.4f} {r["mae"]:>8.4f} {r["cv_mean"]:>10.4f} {r["cv_std"]:>8.4f}')

In [ ]:
# Confusion matrices for top 2 models
top_2 = sorted(results.items(), key=lambda x: -x[1]['accuracy'])[:2]
grade_names = ['F', 'D', 'C', 'B', 'A']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, r) in zip(axes, top_2):
    cm = confusion_matrix(y_test, r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=grade_names, yticklabels=grade_names, ax=ax)
    ax.set_title(f'{name} (Acc: {r["accuracy"]:.3f})')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('figures/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. SHAP Feature Importance

SHAP (SHapley Additive exPlanations) gives us interpretable feature importance that shows both **direction** and **magnitude** of each feature's effect on predictions.

In [ ]:
import shap

# Use XGBoost for SHAP (tree-based, handles multi-class well)
best_tree_name = 'XGBoost'
best_model = results[best_tree_name]['model']
print(f'Computing SHAP values for {best_tree_name}...')

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

print(f'SHAP values shape: {np.array(shap_values).shape}')

In [ ]:
# Global feature importance (mean absolute SHAP across all classes)
shap_arr = np.array(shap_values)
if shap_arr.ndim == 3:
    # Shape: (n_classes, n_samples, n_features) or (n_samples, n_features, n_classes)
    if shap_arr.shape[0] == 5:  # (n_classes, n_samples, n_features)
        mean_shap = np.mean([np.abs(shap_arr[i]).mean(axis=0) for i in range(5)], axis=0)
    else:  # (n_samples, n_features, n_classes)
        mean_shap = np.abs(shap_arr).mean(axis=(0, 2))
else:
    mean_shap = np.abs(shap_arr).mean(axis=0)

importance_df = pd.DataFrame({
    'feature': feature_names,
    'mean_shap': mean_shap
}).sort_values('mean_shap', ascending=True)

fig, ax = plt.subplots(figsize=(10, max(8, len(feature_names) * 0.35)))
ax.barh(importance_df['feature'], importance_df['mean_shap'], color='#2E5599')
ax.set_xlabel('Mean |SHAP Value|')
ax.set_title(f'Feature Importance ({best_tree_name}) — Contextual Features Only')
plt.tight_layout()
plt.savefig('figures/shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP summary plot — what drives an A grade?
shap_arr = np.array(shap_values)
if shap_arr.ndim == 3 and shap_arr.shape[0] == 5:
    # (n_classes, n_samples, n_features) — show class 4 (A grade)
    print('SHAP Summary: What drives an A grade?')
    shap.summary_plot(shap_arr[4], X_test.values, feature_names=feature_names,
                      max_display=20, show=False)
elif shap_arr.ndim == 3:
    # (n_samples, n_features, n_classes)
    print('SHAP Summary: What drives an A grade?')
    shap.summary_plot(shap_arr[:, :, 4], X_test.values, feature_names=feature_names,
                      max_display=20, show=False)
else:
    shap.summary_plot(shap_arr, X_test.values, feature_names=feature_names,
                      max_display=20, show=False)

plt.title(f'SHAP Summary — What Predicts an A Grade ({best_tree_name})')
plt.tight_layout()
plt.savefig('figures/shap_summary_A.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP summary for F grade (what drives failure)
shap_arr = np.array(shap_values)
if shap_arr.ndim == 3 and shap_arr.shape[0] == 5:
    print('SHAP Summary: What drives an F grade?')
    shap.summary_plot(shap_arr[0], X_test.values, feature_names=feature_names,
                      max_display=20, show=False)
elif shap_arr.ndim == 3:
    print('SHAP Summary: What drives an F grade?')
    shap.summary_plot(shap_arr[:, :, 0], X_test.values, feature_names=feature_names,
                      max_display=20, show=False)
else:
    print('Single-output SHAP — skipping per-class plot')

if shap_arr.ndim == 3:
    plt.title(f'SHAP Summary — What Predicts an F Grade ({best_tree_name})')
    plt.tight_layout()
    plt.savefig('figures/shap_summary_F.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Top 10 features with direction of effect
print('\nTop 15 Most Important Contextual Predictors:')
print('=' * 60)
top_features = importance_df.sort_values('mean_shap', ascending=False).head(15)
for _, row in top_features.iterrows():
    feat = row['feature']
    corr_with_grade = df[[feat, 'lg_grade_ord']].dropna().corr().iloc[0, 1]
    direction = 'higher grade' if corr_with_grade > 0 else 'lower grade'
    print(f'  {feat:40s}  SHAP: {row["mean_shap"]:.4f}  (higher value -> {direction})')

## 6. Hyperparameter Tuning (Best Model)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# Tune the best-performing model type
param_dist_xgb = {
    'n_estimators': [300, 500, 700, 1000],
    'max_depth': [4, 5, 6, 7, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.15],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2]
}

xgb_tuned = XGBClassifier(
    objective='multi:softmax', num_class=5,
    random_state=42, n_jobs=-1, verbosity=0
)

search = RandomizedSearchCV(
    xgb_tuned, param_dist_xgb, n_iter=50,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='accuracy', random_state=42, n_jobs=-1, verbose=1
)
search.fit(X_train, y_train)

print(f'\nBest params: {search.best_params_}')
print(f'Best CV accuracy: {search.best_score_:.4f}')

# Evaluate tuned model
y_pred_tuned = search.best_estimator_.predict(X_test)
acc_tuned = accuracy_score(y_test, y_pred_tuned)
mae_tuned = mean_absolute_error(y_test, y_pred_tuned)
print(f'\nTuned Test Accuracy: {acc_tuned:.4f}')
print(f'Tuned MAE: {mae_tuned:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred_tuned, target_names=['F', 'D', 'C', 'B', 'A']))

## 7. Key Findings

In [ ]:
# Generate a clean summary of findings
print('ENHANCED ANALYSIS — KEY FINDINGS')
print('=' * 60)
print()
print('DATA:')
print(f'  Schools: {len(df)} ({len(df_22)} from 2022-23, {len(df_23)} from 2023-24)')
print(f'  Features: {len(feature_names)} contextual (zero formula inputs)')
print(f'  Feature categories: demographics, teacher quality, discipline,')
print(f'    absenteeism, expenditures, staffing, graduation, dropout')
print()
print('MODEL PERFORMANCE (no formula inputs):')
for name, r in sorted(results.items(), key=lambda x: -x[1]['accuracy']):
    print(f'  {name:<25s}  Acc: {r["accuracy"]:.3f}  MAE: {r["mae"]:.3f}  CV: {r["cv_mean"]:.3f}')
if acc_tuned:
    print(f'  {"XGBoost (Tuned)":<25s}  Acc: {acc_tuned:.3f}  MAE: {mae_tuned:.3f}')
print()
print('TOP CONTEXTUAL PREDICTORS:')
for i, (_, row) in enumerate(top_features.iterrows()):
    if i >= 10:
        break
    feat = row['feature']
    corr = df[[feat, 'lg_grade_ord']].dropna().corr().iloc[0, 1]
    direction = '+' if corr > 0 else '-'
    print(f'  {i+1:2d}. {feat:40s} ({direction}) SHAP: {row["mean_shap"]:.4f}')
print()
print('INTERPRETATION:')
print('  These contextual factors predict letter grades WITHOUT using')
print('  any test scores or formula components. This reveals which')
print('  structural conditions are associated with school performance')
print('  ratings, independent of the assessment results themselves.')

In [ ]:
# Save the final merged dataset
output_path = 'enhanced_merged_data.xlsx'
df.to_excel(output_path, index=False)
print(f'Saved enhanced dataset to {output_path}')
print(f'Shape: {df.shape}')